 # 🧾 Day 3 – Conversational Memory Bot

We move from stateless RAG → to stateful conversational systems.

> Before we build a Conversational memory bot, Lets test whether the LLM is capable of remembering the previous conversations.

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0 # pushing it to not be creative
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{input}")
])

chain = prompt | llm

response1 = chain.invoke({"input": "My name is Koushik."})
print(response1.content)

Nice to meet you, Koushik. How can I assist you today?


In [3]:
response2 = chain.invoke({"input": "What is my name?"})
print(response2.content)

I don't have any information about your name. I'm a new conversation each time you interact with me, so I don't retain any personal data or context from previous conversations. If you'd like to share your name with me, I'd be happy to chat with you!


### Observations:

The LLM cannot remember any response from the previous conversation and is strictly limited to context provided.

***LLM = Intelligence component***

***LLM ≠ Application***

Inorder to achieve this, we need to store all the past conversations.

For this, we need to use LangGraph

---
## What LangGraph Is

> LangGraph is a stateful graph engine for LLM workflows — unlike traditional stateless LLM calls, it lets you define a graph of nodes that share and update state, enabling conversational memory and multi-turn behavior.

Key idea:

State = your bot’s memory that persists across invocations
Nodes = functions that update state
Edges = define the flow (order nodes execute in)

In [4]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver

# Define state schema using MessagesState
class ChatState(MessagesState):
    ...

# 1️⃣ Initialize your Groq LLM
llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0)

# 2️⃣ Define node to append user messages
def user_node(state: ChatState):
    # Node returns nothing — state already contains new messages passed in
    return {}

# 3️⃣ Define node to let the LLM reply
def llm_response_node(state: ChatState):
    # LangGraph holds state["messages"] as a growing list of messages
    # We call the LLM with the entire message history so far.
    reply = llm.invoke(state["messages"])
    # Return AI message so MessagesState automatically appends it
    return {"messages": [AIMessage(content=reply.content)]}

# 4️⃣ Build the graph
builder = StateGraph(ChatState)

builder.add_node("user_node", user_node)
builder.add_node("llm_response_node", llm_response_node)

builder.add_edge(START, "user_node")
builder.add_edge("user_node", "llm_response_node")
builder.add_edge("llm_response_node", END)

# Short-term memory via checkpoint
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

# 5️⃣ Chat with memory
session_id = "chat_session_1"
config = {"configurable": {"thread_id": session_id}}

# First user turn
graph.invoke(
    {"messages": [HumanMessage(content="Hi, my name is Alice!")]},
    config,
)

# Second user turn
result = graph.invoke(
    {"messages": [HumanMessage(content="What’s my name?")]},
    config,
)

# Print the agent's reply
print(result["messages"][-1].content)

Your name is Alice.


```
START
  │
  ▼
Load last checkpoint for thread_id
  │
  ▼
Add new HumanMessage to state["messages"]
  │
  ▼
NODE: user_node
  │  (Optional placeholder — state already updated)
  ▼
NODE: llm_response_node
  │  (call Groq LLM with full state["messages"])
  │
  ▼
LLM generates response
  │
  ▼
Append AIMessage to state["messages"]
  │
  ▼
Save updated state as new checkpoint for thread_id
  │
  ▼
Return current state["messages"]
END
```

In [5]:
test_human_messages = [
    HumanMessage(content="Hello!"),
    HumanMessage(content="Hi, can you help me?"),
    HumanMessage(content="What's my name?"),
    HumanMessage(content="Tell me a fun fact about space."),
    HumanMessage(content="How do I cook pasta?"),
    HumanMessage(content="Remind me of my favorite color."),
    HumanMessage(content="What did I say earlier in this conversation?"),
    HumanMessage(content="Can you summarize our chat so far?"),
    HumanMessage(content="I love painting and reading books."),
    HumanMessage(content="My birthday is June 10th."),
]

In [6]:
for msg in test_human_messages:
    result = graph.invoke(
        {"messages": msg.content},
        config,
    )



In [7]:
print(result['messages'])

[HumanMessage(content='Hi, my name is Alice!', additional_kwargs={}, response_metadata={}, id='9bca9f11-436e-4c8a-85ff-e752253a19c1'), AIMessage(content="Nice to meet you, Alice. I'm happy to chat with you. Is there something on your mind that you'd like to talk about, or are you just looking for some company?", additional_kwargs={}, response_metadata={}, id='0f0f32ec-b11c-48da-bca7-b2293d23e642', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What’s my name?', additional_kwargs={}, response_metadata={}, id='64b6cc25-1933-4a71-9d7e-dd461d53f9e0'), AIMessage(content='Your name is Alice.', additional_kwargs={}, response_metadata={}, id='cd0bbfc0-1eb7-46ea-8631-34f0a5edb93f', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Hello!', additional_kwargs={}, response_metadata={}, id='cef772e7-3045-493f-942c-ca77199b4811'), AIMessage(content='Hello Alice! How are you today?', additional_kwargs={}, response_metadata={}, id='09468dd9-296e-408a-939d-760b68e229cb', tool_c

In [8]:
for i in range(len(result["messages"])):
    msg = result["messages"][i]
    if isinstance(msg, HumanMessage):
        role = "user"
    elif isinstance(msg, AIMessage):
        role = "assistant"
    else:
        role = "unknown"
    print(f"{role}: {msg.content}")
    if i%2 != 0: print("\n")

user: Hi, my name is Alice!
assistant: Nice to meet you, Alice. I'm happy to chat with you. Is there something on your mind that you'd like to talk about, or are you just looking for some company?


user: What’s my name?
assistant: Your name is Alice.


user: Hello!
assistant: Hello Alice! How are you today?


user: Hi, can you help me?
assistant: I'd be happy to help you, Alice. What do you need assistance with?


user: What's my name?
assistant: Your name is Alice.


user: Tell me a fun fact about space.
assistant: Here's a fun fact about space: Did you know that there is a giant storm on Jupiter that has been raging for at least 187 years? It's called the Great Red Spot, and it's a massive anticyclonic storm that's so large that three Earths could fit inside it. It's been continuously observed since 1831, and scientists are still trying to figure out what keeps it going for so long.


user: How do I cook pasta?
assistant: Cooking pasta is a simple process. Here's a basic guide:

1. 

### Let's track the token count for the history

In [9]:
# pip install tiktoken
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

def estimate_tokens_from_state(state):
    # approximate: count tokens from all message contents
    total = 0
    for m in state["messages"]:
        total += len(enc.encode(m.content))
    return total

#### Lets check, how many token were in the state

In [10]:
print("Estimated tokens in conversation so far:", estimate_tokens_from_state(result))

Estimated tokens in conversation so far: 936


#### Let's do a new experiment and check how many token were in the state

In [11]:
test_human_messages_2 = [
    HumanMessage(content="Good morning! How are you today?, answer in 1 sentence"),
    HumanMessage(content="Can you suggest a healthy breakfast idea in one word?"),
    HumanMessage(content="Can you remember that I prefer tea over coffee, just say 'yes or no'?"),
    HumanMessage(content="What's something interesting about the ocean, answer in just 1 sentence?"),
    HumanMessage(content="What did I just tell you about my drink preference, in just 1 sentence?")
]
session_id = "chat_session_z"
config = {"configurable": {"thread_id": session_id}}

llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0)

In [12]:
token_c = []
for msg in test_human_messages_2:
    print(msg.content)
    result_2 = graph.invoke(
        {"messages": msg.content},
        config,
    )
    print(result_2["messages"][-1].content)
    token_c.append(estimate_tokens_from_state(result_2))



Good morning! How are you today?, answer in 1 sentence
I'm functioning properly and ready to assist you with any questions or tasks you may have.
Can you suggest a healthy breakfast idea in one word?
Oatmeal.
Can you remember that I prefer tea over coffee, just say 'yes or no'?
Yes.
What's something interesting about the ocean, answer in just 1 sentence?
The ocean is home to the longest mountain range in the world, the Mid-Ocean Ridge, which stretches over 65,000 kilometers.
What did I just tell you about my drink preference, in just 1 sentence?
You mentioned that you prefer tea over coffee.


In [13]:
print(token_c)

[31, 46, 65, 108, 134]


In [14]:
print(result_2)

{'messages': [HumanMessage(content='Good morning! How are you today?, answer in 1 sentence', additional_kwargs={}, response_metadata={}, id='11ab79b3-e8c1-4153-8a12-876732fc3d7c'), AIMessage(content="I'm functioning properly and ready to assist you with any questions or tasks you may have.", additional_kwargs={}, response_metadata={}, id='dc20b6bf-4d9a-4a8d-aa75-2dc38a9b8bd3', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Can you suggest a healthy breakfast idea in one word?', additional_kwargs={}, response_metadata={}, id='cf50b0e7-7e0c-49fb-ac7b-04740da228e7'), AIMessage(content='Oatmeal.', additional_kwargs={}, response_metadata={}, id='2239c6d7-9268-4d80-bf6f-e3680408bbe3', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content="Can you remember that I prefer tea over coffee, just say 'yes or no'?", additional_kwargs={}, response_metadata={}, id='6fc021cd-f676-449f-b489-32a42478b0c3'), AIMessage(content='Yes.', additional_kwargs={}, response_metadata={}, id='eba

In [15]:
for i in range(len(result_2["messages"])):
    msg = result_2["messages"][i]
    if isinstance(msg, HumanMessage):
        role = "user"
    elif isinstance(msg, AIMessage):
        role = "assistant"
    else:
        role = "unknown"
    print(f"{role}: {msg.content}")
    if i%2 != 0:
        print("token_count:", token_c[i//2])
        print("\n")

user: Good morning! How are you today?, answer in 1 sentence
assistant: I'm functioning properly and ready to assist you with any questions or tasks you may have.
token_count: 31


user: Can you suggest a healthy breakfast idea in one word?
assistant: Oatmeal.
token_count: 46


user: Can you remember that I prefer tea over coffee, just say 'yes or no'?
assistant: Yes.
token_count: 65


user: What's something interesting about the ocean, answer in just 1 sentence?
assistant: The ocean is home to the longest mountain range in the world, the Mid-Ocean Ridge, which stretches over 65,000 kilometers.
token_count: 108


user: What did I just tell you about my drink preference, in just 1 sentence?
assistant: You mentioned that you prefer tea over coffee.
token_count: 134




In [16]:
text = "Good morning! How are you today?, answer in 1 sentence  I'm functioning properly and ready to assist you with any questions or tasks you may have"

tks = len(enc.encode(text))
print(tks)

31
